# Huấn luyện mô hình khử nhiễu ECG (Phase 1) trên Google Colab
Notebook này được thiết lập tự động để sao chép mã nguồn, tải dữ liệu cần thiết từ PhysioNet, và lưu checkpoint trực tiếp lên Google Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Tải mã nguồn và cài đặt thư viện

In [ ]:
%cd /content
!git clone https://github.com/vzyhug/phase1.git
%cd phase1
!pip install -r requirements.txt

## 2. Chuẩn bị dữ liệu (QT Database & MIT-BIH NST)
Tải trực tiếp từ PhysioNet và giải nén vào thư mục `data/raw`.

In [ ]:
import os

# Tạo các thư mục dữ liệu
os.makedirs('data/raw/qt_database', exist_ok=True)
os.makedirs('data/raw/mit_bih_nst', exist_ok=True)

# Tải và giải nén QT Database
!wget -q -O qt.zip https://physionet.org/static/published-projects/qtdb/qt-database-1.0.0.zip
!unzip -q -j qt.zip -d data/raw/qt_database

# Tải và giải nén MIT-BIH Noise Stress Test Database
!wget -q -O nst.zip https://physionet.org/static/published-projects/nstdb/mit-bih-noise-stress-test-database-1.0.0.zip
!unzip -q -j nst.zip -d data/raw/mit_bih_nst

!rm qt.zip nst.zip
print("Tải và giải nén dữ liệu hoàn tất.")

## 3. Kết nối thư mục Checkpoints với Google Drive
Bằng cách này, các tệp model checkpoint (như `model.pth`) sẽ được tự động đồng bộ và lưu vào Google Drive, bạn sẽ không bị mất model khi phiên Colab ngắt kết nối.

In [ ]:
import os
drive_ckpt_path = '/content/drive/MyDrive/Phase1_Checkpoints'
os.makedirs(drive_ckpt_path, exist_ok=True)

# Xoá thư mục checkpoints mặc định nếu có và tạo symbolic link tới Drive
!rm -rf checkpoints
!ln -s {drive_ckpt_path} checkpoints
print(f"Thư mục checkpoints đã được liên kết tới {drive_ckpt_path}")

## 4. Tiền xử lý dữ liệu (Preprocessing)
Quá trình này sẽ thực hiện resampling, phân đoạn (segment), chuẩn hóa và tổng hợp dữ liệu nhiễu (synthesize) để tạo file `.npy`.

In [ ]:
!python run_workflow.py --mode preprocess

## 5. Huấn luyện (Training)
Bắt đầu huấn luyện mô hình. Bạn có thể điều chỉnh số epoch, batch size hoặc learning rate trong tệp `configs/base.yaml`.

In [ ]:
!python run_workflow.py --mode train

## 6. Kiểm tra Inference (Tùy chọn)

In [ ]:
!python run_workflow.py --mode infer